# Unified Multi-Output Weather Model (ESP32)

This notebook trains a single Dense (MLP) model to predict next-day precipitation and evapotranspiration from the expanded weather dataset.

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, r2_score
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Input

tf.keras.utils.set_random_seed(42)
np.random.seed(42)


## 2. Load Dataset

In [ ]:
df = pd.read_csv('historical_weather_data.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)
print(f'Loaded {len(df)} daily weather rows.')

## 3. Feature Engineering

In [ ]:
weather_cols = [
    'temperature_2m_max',
    'temperature_2m_min',
    'precipitation',
    'evapotranspiration',
    'shortwave_radiation_sum',
    'soil_moisture_0_to_7cm',
    'relative_humidity_2m',
    'vapor_pressure_deficit',
    'wind_speed_10m',
]

lag_days = range(1, 8)
# Include current-day signals alongside lagged and rolling history.
feature_cols = weather_cols.copy()

for col in weather_cols:
    for lag in lag_days:
        feature_name = f'{col}_lag_{lag}'
        df[feature_name] = df[col].shift(lag)
        feature_cols.append(feature_name)
    for window in (3, 7):
        feature_name = f'{col}_roll_{window}'
        df[feature_name] = df[col].shift(1).rolling(window).mean()
        feature_cols.append(feature_name)

day_of_year = df['time'].dt.dayofyear
df['day_of_year_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
df['day_of_year_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)
feature_cols.extend(['day_of_year_sin', 'day_of_year_cos'])

df['target_precipitation'] = df['precipitation'].shift(-1)
df['target_evapotranspiration'] = df['evapotranspiration'].shift(-1)
df = df.dropna().reset_index(drop=True)

X = df[feature_cols].to_numpy(dtype=np.float32)
# Log-scaling stabilizes the more skewed precipitation target.
y = np.column_stack([
    np.log1p(df['target_precipitation'].to_numpy(dtype=np.float32)),
    df['target_evapotranspiration'].to_numpy(dtype=np.float32),
]).astype(np.float32)

print(f'Feature count: {len(feature_cols)}')
print(f'Input shape: {X.shape}')
print(f'Target shape: {y.shape}')

## 4. Train/Test Split & Scaling

In [ ]:
split_idx = int(len(df) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_std[X_std == 0] = 1e-6

y_mean = y_train.mean(axis=0)
y_std = y_train.std(axis=0)
y_std[y_std == 0] = 1e-6

X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std
y_train_scaled = (y_train - y_mean) / y_std

np.save('X_mean_unified.npy', X_mean)
np.save('X_std_unified.npy', X_std)
np.save('y_mean_unified.npy', y_mean)
np.save('y_std_unified.npy', y_std)

print(f'Training rows: {len(X_train)}')
print(f'Testing rows: {len(X_test)}')

## 5. Unified MLP Architecture

In [ ]:
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.15),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(2),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
)
model.summary()

## 6. Train Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-5),
]

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    verbose=0,
    callbacks=callbacks,
)

print(f"Training stopped after {len(history.history['loss'])} epochs.")
print(f"Best val_loss: {min(history.history['val_loss']):.4f}")

## 7. Performance Evaluation

In [ ]:
pred_scaled = model.predict(X_test_scaled, verbose=0)
pred = (pred_scaled * y_std) + y_mean

pred_precip = np.expm1(pred[:, 0]).clip(min=0)
pred_evapotranspiration = pred[:, 1]

y_test_precip = np.expm1(y_test[:, 0])
y_test_evapotranspiration = y_test[:, 1]

mae_p = mean_absolute_error(y_test_precip, pred_precip)
r2_p = r2_score(y_test_precip, pred_precip)

mae_e = mean_absolute_error(y_test_evapotranspiration, pred_evapotranspiration)
r2_e = r2_score(y_test_evapotranspiration, pred_evapotranspiration)

print('========== PRECIPITATION (Unified) ==========')
print(f'MAE: {mae_p:.3f} mm  | R²: {r2_p:.3f}')
print('\n========== EVAPOTRANSPIRATION (Unified) ==========')
print(f'MAE: {mae_e:.3f} mm  | R²: {r2_e:.3f}')

## 8. Export to TFLite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('unified_weather_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f'Saved unified_weather_model.tflite ({len(tflite_model)} bytes)')

## 9. Generate C Array

In [ ]:
def convert_tflite_to_c_array(tflite_path, c_file_path, array_name):
    with open(tflite_path, 'rb') as f:
        tflite_content = f.read()

    hex_array = [f'0x{b:02x}' for b in tflite_content]

    with open(c_file_path, 'w') as f:
        f.write(f'#include "{array_name}.h"\n\n')
        f.write(f'const unsigned char {array_name}[] = {{\n')
        for i in range(0, len(hex_array), 12):
            f.write('  ' + ', '.join(hex_array[i:i + 12]) + ',\n')
        f.write('};\n\n')
        f.write(f'const int {array_name}_len = {len(hex_array)};\n')

    h_file_path = c_file_path.replace('.cc', '.h')
    with open(h_file_path, 'w') as f:
        f.write(f'#ifndef {array_name.upper()}_H\n')
        f.write(f'#define {array_name.upper()}_H\n\n')
        f.write(f'extern const unsigned char {array_name}[];\n')
        f.write(f'extern const int {array_name}_len;\n\n')
        f.write(f'#endif // {array_name.upper()}_H\n')

    print(f'Generated {c_file_path} and {h_file_path}')

convert_tflite_to_c_array(
    'unified_weather_model.tflite',
    'unified_weather_model.cc',
    'unified_weather_model_tflite',
)